# Forced alignment demo

Align one short audio clip with its known transcript and print the start and end time of every word.

Tool: [stable-ts](https://github.com/jianfch/stable-ts) (`model.align`). Input: `data/sample.wav` and `data/sample.txt`.

**Status:** executed end to end with stable-ts 2.19.1 (see README for the tested setup). The sanity-check cell asserts the result.

In [1]:
# Run this notebook inside the conda env: conda env create -f environment.yml
import shutil
import stable_whisper
print('ffmpeg:', shutil.which('ffmpeg'))

ffmpeg: C:\Users\Mostafa\miniforge3\envs\text-audio-alignment\Library\bin\ffmpeg.EXE


In [2]:
from pathlib import Path

AUDIO = Path('data/sample.wav')
TEXT = Path('data/sample.txt')
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

assert AUDIO.exists(), f'Missing {AUDIO}; see data/SOURCES.md'
assert TEXT.exists(), f'Missing {TEXT}; see data/SOURCES.md'

transcript = TEXT.read_text(encoding='utf-8').strip()
print(transcript)

HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND


In [3]:
import stable_whisper

model = stable_whisper.load_model('base')  # small model; runs on CPU
result = model.align(str(AUDIO), transcript, language='en')

Align:   0%|          | 0/12.49 [00:00<?, ?sec/s]

Align:  95%|█████████▍| 11.86/12.49 [00:02<00:00,  4.12sec/s]

Align: 100%|██████████| 12.49/12.49 [00:02<00:00,  4.34sec/s]

Adjustment:   0%|          | 0/11.86 [00:00<?, ?sec/s]

Adjustment: 100%|██████████| 11.86/11.86 [00:00<00:00, 5908.59sec/s]

In [4]:
import pandas as pd

rows = [
    {'word': w.word.strip(), 'start': round(w.start, 2), 'end': round(w.end, 2)}
    for seg in result.segments
    for w in seg.words
]
df = pd.DataFrame(rows)
df

,word,start,end
0,HE,0.00,0.66
1,TELLS,0.68,1.02
2,US,1.02,1.14
3,THAT,1.14,1.48
4,AT,1.48,1.74
5,THIS,1.74,2.04
6,FESTIVE,2.04,2.52
7,SEASON,2.52,2.94
8,OF,2.94,3.18
9,THE,3.22,3.32


In [5]:
# Sanity checks
n_transcript_words = len(transcript.split())
print('words in transcript:', n_transcript_words)
print('words aligned:      ', len(df))
print('last word ends at:  ', df['end'].max(), 'seconds')
print('start times increasing:', df['start'].is_monotonic_increasing)

assert len(df) == n_transcript_words, 'aligned word count differs from transcript'
assert [w.upper() for w in df['word']] == [w.upper() for w in transcript.split()], 'aligned words differ from transcript'
assert df['start'].is_monotonic_increasing, 'start times are not increasing'
assert (df['end'] >= df['start']).all(), 'a word ends before it starts'
print('sanity checks passed')

words in transcript: 32
words aligned:       32
last word ends at:   11.86 seconds
start times increasing: True
sanity checks passed


In [6]:
# Save outputs
df.to_csv(OUT / 'words.csv', index=False)
result.save_as_json(str(OUT / 'alignment.json'))
result.to_srt_vtt(str(OUT / 'alignment.srt'), word_level=True)
print('saved to', OUT.resolve())

Saved: D:\Claude_workspace\text-audio-alignment\outputs\alignment.json
Saved: D:\Claude_workspace\text-audio-alignment\outputs\alignment.srt
saved to D:\Claude_workspace\text-audio-alignment\outputs
